In [72]:
options(stringsAsFactors = F)
library("CellTrek")
library("dplyr")
library("Seurat")
library("viridis")
library("ConsensusClusterPlus")


ERROR: Error in library("viridis"): 不存在叫‘viridis’这个名称的程序包


In [73]:
library(Seurat)
library(Matrix)

# ============================================================
# 1. 读取 ST 数据
# ============================================================
st_mat <- read.table(
    "/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/st_expressionx.tsv",
    header = TRUE, sep = "\t", row.names = 1, check.names = FALSE
)
st_coords <- read.table(
    "/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/spatial_coords.tsv",
    header = TRUE, sep = "\t", row.names = 1, check.names = FALSE
)
rownames(st_coords)=colnames(st_mat) 
# 确保坐标行名与表达矩阵列名一致
stopifnot(all(colnames(st_mat) == rownames(st_coords)))
colnames(st_coords)=c('spatial_1','spatial_2')

# 创建 Seurat 对象
st_obj <- CreateSeuratObject(counts = as.matrix(st_mat), project = "ST")
st_obj[["spatial"]] <- CreateDimReducObject(
    embeddings = as.matrix(st_coords),
    key = "spatial_",
    assay = DefaultAssay(st_obj)
)
# 也可以放到 meta.data 中方便查看
st_obj$x <- st_coords[, 1]
st_obj$y <- st_coords[, 2]

print(st_obj)

# ============================================================
# 2. 读取 sc 数据
# ============================================================
sc_mat <- read.table(
    "/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/sc_expression.tsv",
    header = TRUE, sep = "\t", row.names = 1, check.names = FALSE
)
cell_labels <- read.table(
    "/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/decov/cell_type_labels.tsv",
    header = TRUE, sep = "\t", 
    row.names = 1, check.names = FALSE,
    colClasses = "character"      # 强制所有列为字符
)

# 获取标签向量（假设标签文件有两列：cell_id 和 label，或者只有一列）
if (ncol(cell_labels) == 1) {
    labels_vec <- cell_labels[[1]]
    names(labels_vec) <- rownames(cell_labels)
} else {
    # 找到包含细胞类型标签的那一列
    labels_vec <- cell_labels[[1]]
    names(labels_vec) <- rownames(cell_labels)
}





Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”


An object of class Seurat 
254 features across 424 samples within 1 assay 
Active assay: RNA (254 features, 0 variable features)
 1 layer present: counts
 1 dimensional reduction calculated: spatial


In [74]:
# 确保细胞顺序一致
common_cells <- intersect(colnames(sc_mat), names(labels_vec))
sc_mat <- sc_mat[, common_cells]
labels_vec <- labels_vec[common_cells]

In [75]:
# 创建 Seurat 对象
sc_obj <- CreateSeuratObject(counts = as.matrix(sc_mat), project = "sc")
sc_obj$cell_type <- labels_vec

print(sc_obj)

# ============================================================
# 3. 可选：保存
# ============================================================
saveRDS(st_obj, "/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/CellTrek/st_obj.rds")
saveRDS(sc_obj, "/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/SimMerfish/CellTrek/sc_obj.rds")

Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”


An object of class Seurat 
254 features across 4198 samples within 1 assay 
Active assay: RNA (254 features, 0 variable features)
 1 layer present: counts


In [76]:
traint1=function (st_data, sc_data, st_assay = "Spatial", sc_assay = "scint", 
    norm = "LogNormalize", nfeatures = 2000, cell_names = "cell_names", 
    coord_xy = c("imagerow", "imagecol"), coord_df = NULL, gene_kept = NULL, 
    ...) 
{
    st_data$id <- names(st_data$orig.ident)
    sc_data$id <- names(sc_data$orig.ident)
    sc_data$cell_names <- make.names(sc_data@meta.data[, cell_names])
    st_data$type <- "st"
    sc_data$type <- "sc"
    st_cells <- Cells(st_data)
    if (!is.null(coord_df)) {
        coord_df <- as.data.frame(coord_df)
        if (all(coord_xy %in% colnames(coord_df))) {
            xy <- coord_df[, coord_xy, drop = FALSE]
        }
        else {
            if (ncol(coord_df) < 2) 
                stop("coord_df must have at least two columns (coord_x, coord_y)")
            xy <- coord_df[, 1:2, drop = FALSE]
        }
        if (!all(st_cells %in% rownames(xy))) {
            stop("coord_df rownames must cover all ST cells/spots (Cells(st_data)); ", 
                sum(!st_cells %in% rownames(xy)), " missing")
        }
        st_data$coord_x <- xy[st_cells, 1]
        st_data$coord_y <- xy[st_cells, 2]
    }
    else if (length(st_data@images) > 0 && "coordinates" %in% 
        slotNames(st_data@images[[1]]) && all(coord_xy %in% colnames(st_data@images[[1]]@coordinates))) {
        st_coord <- st_data@images[[1]]@coordinates
        st_data$coord_x <- st_coord[st_cells, coord_xy[1]]
        st_data$coord_y <- st_coord[st_cells, coord_xy[2]]
    }
    else {
        st_coord <- tryCatch(Seurat::GetTissueCoordinates(st_data), 
            error = function(e) NULL)
        if (is.null(st_coord)) {
            stop("Could not read spatial coordinates from st_data (unsupported image structure). ", 
                "Please supply them via the 'coord_df' argument.")
        }
        if (all(coord_xy %in% colnames(st_coord))) {
            st_data$coord_x <- st_coord[st_cells, coord_xy[1]]
            st_data$coord_y <- st_coord[st_cells, coord_xy[2]]
        }
        else if (all(c("x", "y") %in% colnames(st_coord))) {
            st_data$coord_x <- st_coord[st_cells, "y"]
            st_data$coord_y <- st_coord[st_cells, "x"]
        }
        else {
            stop("Spatial coordinate columns not found (looked for c('", 
                coord_xy[1], "','", coord_xy[2], "') or c('x','y')). Please supply them via the 'coord_df' argument.")
        }
    }
    if (anyNA(st_data$coord_x) || anyNA(st_data$coord_y)) {
        stop("Some ST cells/spots have missing coordinates after resolution; check coord_df / image coordinates.")
    }
    DefaultAssay(st_data) <- st_assay
    DefaultAssay(sc_data) <- sc_assay
    cat("Finding transfer anchors... \n")
    st_idx <- st_data$id
    sc_idx <- sc_data$id
    sc_st_list <- list(st_data = st_data, sc_data = sc_data)
    sc_st_features <- Seurat::SelectIntegrationFeatures(sc_st_list, 
        nfeatures = nfeatures)
    if (!is.null(gene_kept)) {
        sc_st_features <- union(sc_st_features, gene_kept)
    }
    sc_st_features <- sc_st_features[(sc_st_features %in% rownames(GetAssayData(st_data, 
        assay = st_assay, layer = "data"))) & (sc_st_features %in% 
        rownames(GetAssayData(sc_data, assay = sc_assay, layer = "data")))]
    cat("Using", length(sc_st_features), "features for integration... \n")
    sc_st_anchors <- Seurat::FindTransferAnchors(reference = sc_data, 
        query = st_data, reference.assay = sc_assay, query.assay = st_assay, 
        normalization.method = norm, features = sc_st_features, 
        reduction = "cca", ...)
    cat("Data transfering... \n")
    st_data_trans <- Seurat::TransferData(anchorset = sc_st_anchors, 
        refdata = GetAssayData(sc_data, assay = sc_assay, layer = "data")[sc_st_features, 
            ], weight.reduction = "cca")
    st_data@assays$transfer <- st_data_trans
    cat("Creating new Seurat object... \n")
    sc_st_meta <- dplyr::bind_rows(st_data@meta.data, sc_data@meta.data)
    counts_temp <- cbind(data.frame(GetAssayData(st_data, assay = "transfer", 
        layer = "data")), data.frame(GetAssayData(sc_data, assay = sc_assay, 
        layer = "data")[sc_st_features, ] %>% data.frame))
    rownames(sc_st_meta) <- make.names(sc_st_meta$id)
    colnames(counts_temp) <- make.names(sc_st_meta$id)
    sc_st_int <- CreateSeuratObject(counts = as.matrix(counts_temp), 
        assay = "traint", meta.data = sc_st_meta)
    sc_st_int <- SetAssayData(sc_st_int, assay = "traint", layer = "data", 
        new.data = GetAssayData(sc_st_int, assay = "traint", 
            layer = "counts"))
    cat("Scaling -> PCA -> UMAP... \n")
    sc_st_int <- ScaleData(sc_st_int, features = sc_st_features) %>% 
        RunPCA(features = sc_st_features)
    sc_st_int <- RunUMAP(sc_st_int, dims = 1:30)
    return(sc_st_int)
}

In [77]:
st_obj <- NormalizeData(st_obj, normalization.method = "LogNormalize", scale.factor = 100)
sc_obj <- NormalizeData(sc_obj, normalization.method = "LogNormalize", scale.factor = 100)

Normalizing layer: counts

Normalizing layer: counts



In [ ]:
brain_traint <- traint1(
    st_data    = st_obj,
    sc_data    = sc_obj,
    st_assay   = "RNA",
    sc_assay   = "RNA",
    cell_names = "cell_type",     # sc_obj 中细胞类型列
    coord_df=st_coords,
    coord_xy   = c("spatial_1", "spatial_2")      # st_obj 中坐标列
)


Finding transfer anchors... 


No variable features found for object1 in the object.list. Running FindVariableFeatures ...

Finding variable features for layer counts

No variable features found for object2 in the object.list. Running FindVariableFeatures ...

Finding variable features for layer counts



Using 254 features for integration... 


Running CCA

Merging objects

Finding neighborhoods

Finding anchors

	Found 2053 anchors



Data transfering... 


Finding integration vectors

Finding integration vector weights

Transfering 254 features onto reference data



Creating new Seurat object... 


Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”


Scaling -> PCA -> UMAP... 


Centering and scaling data matrix

PC_ 1 
Positive:  Slc17a7, Satb2, Slc30a3, C1ql3, Wipf3, Syndig1, Unc5d, Prdm8, Ptprt, Col25a1 
	   Ptprk, Meis2, Nr4a1, Lamp5, Epha7, Cux2, Rgs6, Cbln2, Grm1, Npnt 
	   Grm8, Ccdc3, Otof, Cdh12, Sertm1, Ntng2, Shisa9, Ccbe1, Prr16, Dscaml1 
Negative:  Timp3, Cldn5, Flt1, Vtn, Bgn, Rgs5, Igf2, Cobll1, Plekhg3, Nr2f2 
	   Ptprm, Unc5b, Daam2, Gfap, Cspg4, Cxcl14, Pdlim5, Aqp4, Tbc1d4, Serpinf1 
	   Myh14, Kcnj8, Cd14, Sox10, Egfr, Lama3, Tshz2, Tmtc2, Pxdc1, Pdgfra 
PC_ 2 
Positive:  Bcl11b, Rab3b, Sulf2, Spon1, Slc32a1, Syt6, Tox, Gad1, Fezf2, Sema5a 
	   Foxp2, Prss23, Parm1, Grik1, Rnf152, Alk, Nxph1, Lhx6, Elfn1, Reln 
	   Ubash3b, Ano4, Plcxd3, Grm8, Pvalb, Adra1b, Prss12, Cnr1, Sst, Ptpru 
Negative:  Cux2, Lamp5, Calb1, Otof, Ptprk, Unc5d, Slc30a3, Igfbp5, Itgb8, Timp3 
	   C1ql3, Gpc6, Prdm8, Sertm1, Flt1, Cldn5, Rgs5, Igf2, Rorb, Ccbe1 
	   Vtn, Bgn, Ccdc3, Gfap, Trp53i11, Aqp4, Cdh12, Rgs6, Cobll1, Trpc6 
PC_ 3 
Positive:  Fezf2, Slc17a7, Tox,

In [124]:
celltrek1=function (st_sc_int, int_assay = "traint", sc_data = NULL, sc_assay = "RNA", 
    reduction = "pca", intp = T, intp_pnt = 10000, intp_lin = F, 
    nPCs = 30, ntree = 1000, dist_thresh = 0.4, top_spot = 10, 
    spot_n = 10, repel_r = 5, repel_iter = 10, keep_model = F, 
    ...) 
{
    dist_res <- CellTrek:::celltrek_dist(st_sc_int = st_sc_int, int_assay = int_assay, 
        reduction = reduction, intp = intp, intp_pnt = intp_pnt, 
        intp_lin = intp_lin, nPCs = nPCs, ntree = ntree, keep_model = T)
    spot_dis_intp <- median(unlist(dbscan::kNN(dist_res$coord_df[, 
        c("coord_x", "coord_y")], k = 4)$dist))
    if (is.null(repel_r)) {
        repel_r = spot_dis_intp/4
    }
    sc_coord_list <- CellTrek:::celltrek_chart(dist_mat = dist_res$celltrek_dist, 
        coord_df = dist_res$coord_df, dist_cut = ntree * dist_thresh, 
        top_spot = top_spot, spot_n = spot_n, repel_r = repel_r, 
        repel_iter = repel_iter)
    sc_coord_raw <- sc_coord_list[[1]]
    sc_coord <- sc_coord_list[[2]]
    cat("Creating Seurat Object... \n")
    if (!is.null(sc_data)) {
        cat("sc data...")
        sc_data$id <- Seurat::Cells(sc_data)
        sc_out <- CreateSeuratObject(counts = as.matrix(GetAssayData(sc_data, 
            assay = sc_assay, layer = "data")[, sc_coord$id_raw]) %>% 
            magrittr:::set_colnames(sc_coord$id_new), project = "celltrek", 
            assay = sc_assay, meta.data = sc_data@meta.data[sc_coord$id_raw, 
                ] %>% dplyr::rename(id_raw = id) %>% mutate(id_new = sc_coord$id_new) %>% 
                magrittr:::set_rownames(sc_coord$id_new))
        sc_out@meta.data <- dplyr::left_join(sc_out@meta.data, 
            sc_coord) %>% data.frame %>% magrittr:::set_rownames(sc_out$id_new)
        sc_out <- SetAssayData(sc_out, assay = sc_assay, layer = "data", 
            new.data = GetAssayData(sc_out, assay = sc_assay, 
                layer = "counts"))
        sc_coord_raw_df <- CreateDimReducObject(embeddings = sc_coord_raw %>% 
            dplyr::mutate(coord1 = coord_y, coord2 = max(coord_x) + 
                min(coord_x) - coord_x) %>% dplyr::select(c(coord1, 
            coord2)) %>% magrittr:::set_rownames(sc_coord_raw$id_new) %>% 
            as.matrix, assay = sc_assay, key = "celltrekraw_")
        sc_coord_dr <- CreateDimReducObject(embeddings = sc_coord %>% 
            dplyr::mutate(coord1 = coord_y, coord2 = max(coord_x) + 
                min(coord_x) - coord_x) %>% dplyr::select(c(coord1, 
            coord2)) %>% magrittr:::set_rownames(sc_coord$id_new) %>% as.matrix, 
            assay = sc_assay, key = "celltrek_")
        sc_out@reductions$celltrek <- sc_coord_dr
        sc_out@reductions$celltrek_raw <- sc_coord_raw_df
        if ("pca" %in% names(sc_data@reductions)) {
            sc_pca_dr <- CreateDimReducObject(embeddings = sc_data@reductions$pca@cell.embeddings[sc_coord$id_raw, 
                ] %>% magrittr:::set_rownames(sc_coord$id_new) %>% as.matrix, 
                assay = sc_assay, key = "pca_")
            sc_out@reductions$pca <- sc_pca_dr
        }
        if ("umap" %in% names(sc_data@reductions)) {
            sc_umap_dr <- CreateDimReducObject(embeddings = sc_data@reductions$umap@cell.embeddings[sc_coord$id_raw, 
                ] %>% magrittr:::set_rownames(sc_coord$id_new) %>% as.matrix, 
                assay = sc_assay, key = "umap_")
            sc_out@reductions$umap <- sc_umap_dr
        }
        if ("tsne" %in% names(sc_data@reductions)) {
            sc_tsne_dr <- CreateDimReducObject(embeddings = sc_data@reductions$tsne@cell.embeddings[sc_coord$id_raw, 
                ] %>% magrittr:::set_rownames(sc_coord$id_new) %>% as.matrix, 
                assay = sc_assay, key = "tsne_")
            sc_out@reductions$tsne <- sc_tsne_dr
        }
    }
    else {
        cat("no sc data...")
        sc_out <- CreateSeuratObject(counts = as.matrix(GetAssayData(st_sc_int, 
            assay = int_assay, layer = "data")[, sc_coord$id_raw]) %>% 
            set_colnames(sc_coord$id_new), project = "celltrek", 
            assay = int_assay, meta.data = st_sc_int@meta.data[sc_coord$id_raw, 
                ] %>% dplyr::rename(id_raw = id) %>% mutate(id_new = sc_coord$id_new) %>% 
                magrittr:::set_rownames(sc_coord$id_new))
        sc_out$coord_x <- sc_coord$coord_x[match(sc_coord$id_new, 
            sc_out$id_new)]
        sc_out$coord_y <- sc_coord$coord_y[match(sc_coord$id_new, 
            sc_out$id_new)]
        sc_out <- SetAssayData(sc_out, assay = int_assay, layer = "data", 
            new.data = GetAssayData(sc_out, assay = int_assay, 
                layer = "counts"))
        sc_out <- SetAssayData(sc_out, assay = int_assay, layer = "scale.data", 
            new.data = GetAssayData(st_sc_int, assay = int_assay, 
                layer = "scale.data")[, sc_coord$id_raw] %>% 
                set_colnames(sc_coord$id_new))
        sc_coord_raw_df <- CreateDimReducObject(embeddings = sc_coord_raw %>% 
            dplyr::mutate(coord1 = coord_y, coord2 = max(coord_x) + 
                min(coord_x) - coord_x) %>% dplyr::select(c(coord1, 
            coord2)) %>% magrittr:::set_rownames(sc_coord_raw$id_new) %>% 
            as.matrix, assay = sc_assay, key = "celltrekraw_")
        sc_coord_dr <- CreateDimReducObject(embeddings = sc_coord %>% 
            dplyr::mutate(coord1 = coord_y, coord2 = max(coord_x) + 
                min(coord_x) - coord_x) %>% dplyr::select(c(coord1, 
            coord2)) %>% magrittr:::set_rownames(sc_coord$id_new) %>% as.matrix, 
            assay = int_assay, key = "celltrek_")
        sc_out@reductions$celltrek <- sc_coord_dr
        sc_out@reductions$celltrek_raw <- sc_coord_raw_df
        if ("pca" %in% names(st_sc_int@reductions)) {
            sc_pca_dr <- CreateDimReducObject(embeddings = st_sc_int@reductions$pca@cell.embeddings[sc_coord$id_raw, 
                ] %>% magrittr:::set_rownames(sc_coord$id_new) %>% as.matrix, 
                assay = int_assay, key = "pca_")
            sc_out@reductions$pca <- sc_pca_dr
        }
        if ("umap" %in% names(st_sc_int@reductions)) {
            sc_umap_dr <- CreateDimReducObject(embeddings = st_sc_int@reductions$umap@cell.embeddings[sc_coord$id_raw, 
                ] %>% magrittr:::set_rownames(sc_coord$id_new) %>% as.matrix, 
                assay = int_assay, key = "umap_")
            sc_out@reductions$umap <- sc_umap_dr
        }
        if ("tsne" %in% names(st_sc_int@reductions)) {
            sc_tsne_dr <- CreateDimReducObject(embeddings = st_sc_int@reductions$tsne@cell.embeddings[sc_coord$id_raw, 
                ] %>% magrittr:::set_rownames(sc_coord$id_new) %>% as.matrix, 
                assay = int_assay, key = "tsne_")
            sc_out@reductions$tsne <- sc_tsne_dr
        }
    }
    sc_out@images <- st_sc_int@images
    #sc_out@images[[1]]@assay <- DefaultAssay(sc_out)
    #if ("coordinates" %in% slotNames(sc_out@images[[1]])) {
    #    sc_out@images[[1]]@coordinates <- data.frame(imagerow = sc_coord$coord_x, 
    #        imagecol = sc_coord$coord_y) %>% set_rownames(sc_coord$id_new)
    #}
    #sc_out@images[[1]]@scale.factors$spot_dis <- dist_res$spot_d
    #sc_out@images[[1]]@scale.factors$spot_dis_intp <- spot_dis_intp
    output <- list(celltrek = sc_out)
    if (keep_model) {
        output[[length(output) + 1]] <- dist_res$model
        names(output)[length(output)] <- "model"
    }
    return(output)
}

In [ ]:
new_names <- paste0("X", colnames(sc_obj))
sc_obj <- RenameCells(sc_obj, new.names = new_names)


Distance between spots is: 90 
Interpolating...


Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z

Random Forest training... 
Random Forest prediction...  
Making distance matrix... 
Making graph... 
Pruning graph...


Joining with `by = join_by(Var1, Var2, value, val_rsc, Var1_type, Var2_type)`


Spatial Charting SC data...
Repelling points...
Creating Seurat Object... 
sc data...

Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Joining with `by = join_by(id_raw, id_new)`


ERROR: Error in `*tmp*`[[1]]: 下标出界


In [ ]:
brain_celltrek <- celltrek1(st_sc_int=brain_traint, int_assay='traint', sc_data=sc_obj, sc_assay = 'RNA', 
                                   reduction='pca', intp=T, intp_pnt=5000, intp_lin=F, nPCs=30, ntree=1000, 
                                   dist_thresh=0.55, top_spot=5, spot_n=5, repel_r=5, repel_iter=20, keep_model=T)$celltrek

Distance between spots is: 90 
Interpolating...


Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z, linear, input, output, kernel, h, :
“triangle collapsed!”
Warning message in interpShull(xo, yo, x, y, z

Random Forest training... 
Random Forest prediction...  
Making distance matrix... 
Making graph... 
Pruning graph...


Joining with `by = join_by(Var1, Var2, value, val_rsc, Var1_type, Var2_type)`


Spatial Charting SC data...
Repelling points...
Creating Seurat Object... 
sc data...

Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Joining with `by = join_by(id_raw, id_new)`


In [ ]:
rownames0=intersect(rownames(brain_celltrek@meta.data), colnames(sc_obj))
newdata=brain_celltrek@meta.data[rownames0,c(7,8)]

In [ ]:
# 假设 newdata 和 st_coords 已经加载
# newdata: 行名为长字符串，列 coord_x, coord_y
# st_coords: 行名为 X0, X1, ...，列 spatial_1, spatial_2

# 提取坐标矩阵
new_xy <- as.matrix(newdata[, c("coord_x", "coord_y")])
st_xy  <- as.matrix(st_coords[, c("spatial_1", "spatial_2")])

# 计算距离矩阵（欧氏距离）
# 利用矩阵运算加速：d^2 = |a|^2 + |b|^2 - 2 a·b
new_sq <- rowSums(new_xy^2)
st_sq  <- rowSums(st_xy^2)
cross  <- new_xy %*% t(st_xy)
dmat   <- sqrt(outer(new_sq, rep(1, nrow(st_xy))) + 
               outer(rep(1, nrow(new_xy)), st_sq) - 
               2 * cross)

# 对每个 newdata 点找到最近邻索引
nn_idx <- apply(dmat, 1, which.min)

# 构建结果 data.frame
result <- data.frame(
    new_id    = rownames(newdata),
    new_x     = newdata$coord_x,
    new_y     = newdata$coord_y,
    st_id     = rownames(st_coords)[nn_idx],
    st_x      = st_coords$spatial_1[nn_idx],
    st_y      = st_coords$spatial_2[nn_idx],
    stringsAsFactors = FALSE
)

# 查看前几行
head(result)

An object of class Seurat 
254 features across 424 samples within 1 assay 
Active assay: RNA (254 features, 0 variable features)
 2 layers present: counts, data
 1 dimensional reduction calculated: spatial

In [170]:
# 假设 result 是上一步生成的 data.frame，包含 new_id 和 st_id
# 确保 new_id 的顺序与 newdata 的行顺序一致
new_ids <- rownames(newdata)  # 所有 new_id
st_ids <- rownames(st_coords) # 所有 st_id

# 构建 0/1 矩阵
mat <- matrix(0, nrow = length(new_ids), ncol = length(st_ids),
              dimnames = list(new_ids, st_ids))

# 找到 result 中每个 new_id 对应的行索引和 st_id 对应的列索引
row_idx <- match(result$new_id, new_ids)
col_idx <- match(result$st_id, st_ids)

# 赋值 1
mat[cbind(row_idx, col_idx)] <- 1

# 查看维度
dim(mat)  # 应为 length(new_ids) x length(st_ids)

[1] 1900  424

In [174]:
write.table(mat,'CellTreck_assignment.txt',quote=F)